In [209]:
import numpy as np

In [ ]:
class Linear:
    def __init__(self, in_features, out_features, bias=True):
        self.in_features = in_features
        self.out_features = out_features
        # std = np.sqrt(2.0 / (self.in_features + self.out_features))
        # self.weights = np.random.normal(0, 1, (out_features, in_features)) * std # <- xavier init
        std = np.sqrt(2.0 / in_features)
        self.weight = np.random.normal(0, 1, (out_features, in_features)) * std # <- kaiming init
        self.bias = np.zeros(out_features) if bias else None

        self.grad_weight = np.zeros((out_features, in_features))
        self.grad_bias = np.zeros(out_features) if bias else None
    def __call__(self, x):
        self.input = x
        output = self.input @ self.weight.T 
        if self.bias is not None:
            output += self.bias
        return output
    def backward(self, grad_output):
        self.grad_weight = grad_output.T @ self.input
        if self.bias is not None:
            self.grad_bias = grad_output.sum(axis=0)
        grad_input = grad_output @ self.weight
        return grad_input
    def parameters(self):
        params = [[self.weight, self.grad_weight]]
        if self.bias is not None:
            params.append([self.bias, self.grad_bias])
        return params

class Softmax:
    def __call__(self, logits):
        exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        self.probs = exps / np.sum(exps, axis=1, keepdims=True)
        return self.probs
    def backward(self, grad_output):
        return self.probs * (grad_output - (grad_output*self.probs).sum(axis=1, keepdims=True))
    def parameters(self):
        return []
        
class Sigmoid:  
    def __call__(self, logits):
        self.output = 1 / (1 + np.exp(-logits))
        return self.output
    def backward(self, grad_output):
        return grad_output * self.output * (1 - self.output)
    def parameters(self):
        return []
class ReLU:
    def __call__(self, logits):
        self.input = logits
        return np.maximum(0, logits)
    def backward(self, grad_output):
        return grad_output * (self.input > 0)
    def parameters(self):
        return []

In [ ]:
class L2Loss:
    def __call__(self, target, prediction):
        self.target = target
        self.prediction = prediction
        return 1/2 * np.sum((target-prediction)**2)
    def backward(self):
        return self.prediction - self.target

class CrossEntropy:
    def __call__(self, probs, target):
        self.target = target
        self.probs = probs
        return -np.sum(target*np.log(probs + 1e-9)) / probs.shape[0]
    def backward(self):
        self.grad = -(self.target/ (self.probs  + 1e-9)) / self.probs.shape[0]
        return self.grad


class SoftmaxCrossEntropy:
    def __call__(self, logits, target):
        exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        self.probs = exps / np.sum(exps, axis=1, keepdims=True)
        self.target = target
        self.batch_size = logits.shape[0]
        return -np.sum(target*np.log(self.probs)) / self.batch_size
    def backward(self):
        return (self.probs - self.target) / self.batch_size

In [423]:
class Net:
    def __init__(self, layers):
        self.layers = layers
    def forward(self, x):
        self.input = x
        for layer in self.layers:
            self.input = layer(self.input)
        return self.input  
    def backward(self, grad_output):
        for layer in reversed(self.layers):
            grad_output = layer.backward(grad_output)
    def __call__(self, x):
        return self.forward(x)
    def parameters(self):
        return [parameter for layer in self.layers for parameter in layer.parameters()]

In [424]:
net = Net([
    Linear(3, 2, bias=False), 
    Sigmoid(),
    Linear(2, 2),
    Sigmoid(),
])

In [435]:
net.layers[0].weight = np.array([[0.1, -0.2, 0.3], [-0.4, 0.5, -0.6]])
net.layers[2].weight = np.array([[-0.25, 0.35], [0.55, -0.65]])
net.layers[2].bias = np.array([0.15, -0.45])

In [436]:
pred = net([[1, 1, 0]])

In [437]:
loss = L2Loss()

In [438]:
y = np.array([1, 0])
L = loss(y, pred)
grad_L = loss.backward()

In [439]:
net.backward(grad_L)

In [440]:
net.layers[0].grad_bias

In [441]:
net.parameters()

[[array([[ 0.1, -0.2,  0.3],
         [-0.4,  0.5, -0.6]]),
  array([[ 0.01873169,  0.01873169,  0.        ],
         [-0.02363827, -0.02363827,  0.        ]])],
 [array([[-0.25,  0.35],
         [ 0.55, -0.65]]),
  array([[-0.05241141, -0.05792356],
         [ 0.04105087,  0.04536823]])],
 [array([ 0.15, -0.45]), array([-0.11033497,  0.0864191 ])]]

In [442]:
net.parameters()[0][0] -= net.parameters()[0][1]

In [443]:
net.parameters()

[[array([[ 0.08126831, -0.21873169,  0.3       ],
         [-0.37636173,  0.52363827, -0.6       ]]),
  array([[ 0.01873169,  0.01873169,  0.        ],
         [-0.02363827, -0.02363827,  0.        ]])],
 [array([[-0.25,  0.35],
         [ 0.55, -0.65]]),
  array([[-0.05241141, -0.05792356],
         [ 0.04105087,  0.04536823]])],
 [array([ 0.15, -0.45]), array([-0.11033497,  0.0864191 ])]]

In [355]:
lin1 = Linear(3, 2, bias=False)
lin2 = Linear(2, 2)
sig1 = Sigmoid()
sig2 = Sigmoid()
relu = ReLU()
lin1.weight = np.array([[0.1, -0.2, 0.3], [-0.4, 0.5, -0.6]])
lin2.weight = np.array([[-0.25, 0.35], [0.55, -0.65]])
lin2.bias = np.array([0.15, -0.45])
softmax = Softmax()
loss = L2Loss()

In [356]:
nett = lin1(np.array([[1, 1, 0]]))
print(nett)

[[-0.1  0.1]]


In [357]:
a = sig2(lin2(sig1(nett)))

In [358]:
a

array([[0.55354082, 0.37052271]])

In [359]:
y = np.array([1, 0])

In [360]:
print(loss(y, a))

0.1683064414643094


In [361]:
l1 = loss.backward()
l1

array([[-0.44645918,  0.37052271]])

In [362]:
l2 = sig2.backward(l1)
l2

array([[-0.11033497,  0.0864191 ]])

In [363]:
l3 = lin2.backward(l2)
l3

array([[ 0.07511425, -0.09478965]])

In [364]:
lin2.grad_weight

array([[-0.05241141, -0.05792356],
       [ 0.04105087,  0.04536823]])

In [365]:
l4 = sig1.backward(l3)
l4

array([[ 0.01873169, -0.02363827]])

In [366]:
l5 = lin1.backward(l4)
l5

array([[ 0.01132848, -0.01556547,  0.01980247]])

In [367]:
lin1.grad_weight

array([[ 0.01873169,  0.01873169,  0.        ],
       [-0.02363827, -0.02363827,  0.        ]])